## Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns
import json
import tiktoken
from chunking_evaluation.chunking import RecursiveTokenChunker
from chunking_util import load_cuad_df, build_clean_df, chunk_text_with_offsets, classify_spans

from cuad_cleaning import clean_text_with_map

## Loading Data

In [28]:
def find_cuad_data_path(data_set_path: str) -> Path:
    """Resolve a CUAD data file relative to the repo's data/cuad directory."""
    target_path = Path(os.getcwd()).parent / 'data' / 'cuad' / data_set_path #curr dir, go to parent, cd into dataset of choice
    if not target_path.exists():
        raise FileNotFoundError(f"CUAD data file not found: {target_path}")
    return target_path
target_path = find_cuad_data_path('train_separate_questions.json')

In [ ]:
df = load_cuad_df(target_path)

## Data Cleaning Pipeline

In [ ]:
# text-cleaning + the full clean_df build now live in chunking_util.py (imported above),
# so this notebook doesn't keep its own separate copy of that logic anymore

In [ ]:
clean_df = build_clean_df(target_path)

In [33]:
# double-check nothing got silently mangled while cleaning, separately re-clean each raw answer
# on its own and make sure it still matches what we ended up with. if a row ever doesn't match,
# it means the cleanup accidentally ate part of a real clause instead of just boilerplate.
has_answer = ~clean_df["is_impossible"]

independent_clean = df["answer_text"].apply(lambda s: clean_text_with_map(s)[0] if isinstance(s, str) else None)
mismatch_mask = has_answer & (clean_df["annotation_text"].fillna("") != independent_clean.fillna(""))

flagged = clean_df[mismatch_mask]
print(f"total flagged: {len(flagged)} / {has_answer.sum()} answered rows")
if len(flagged) > 0:
    print(flagged[["contract_id", "category", "annotation_text"]].head(10))
    raise ValueError(
        f"{len(flagged)} answer(s) don't survive independent re-cleaning -- offset relocation likely "
        f"ate part of a real clause. Do not trust clean_df until this is fixed; see rows above."
    )

total flagged: 0 / 11180 answered rows


## Chunking Pipeline

Settled config (see `alec-eda-01.ipynb` for chunker choice and
`chunk_size_sweep.py` / `check_768_config.py` for the window-size
sweep and full-corpus validation behind these numbers): `RecursiveTokenChunker`,
**768 tokens, 192-token overlap, `cl100k_base`**. At this window, 0 evidence
spans are OVERSIZED and 99.7% are fully PRESERVED inside one chunk, using 38%
fewer chunks than the smaller 512/128 config first tried.

Offset recovery (`chunk_text_with_offsets`) is imported from
`chunking_util.py` rather than reimplemented here — see that
module's docstring for why it searches the raw contract text directly instead
of a whitespace-normalized copy.

In [34]:
WINDOW_SIZE = 768
OVERLAP = 192
ENCODING_NAME = "cl100k_base"

encoding = tiktoken.get_encoding(ENCODING_NAME)

def token_len(text: str) -> int:
    return len(encoding.encode(text))

chunker = RecursiveTokenChunker(
    chunk_size=WINDOW_SIZE,
    chunk_overlap=OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=token_len,
)

In [9]:
contracts = clean_df.drop_duplicates("contract_id")[["contract_id", "text"]]
rows = []
for contract_id, text in contracts.itertuples(index=False):
    for i, (chunk_text, (start, end)) in enumerate(chunk_text_with_offsets(chunker, text)):
        rows.append({
            "contract_id": contract_id,
            "chunk_id": f"{contract_id}__chunk{i}",
            "chunk_index": i,
            "chunk_text": chunk_text,
            "chunk_start": start,
            "chunk_end": end,
        })
chunks_df = pd.DataFrame(rows)

n_offset_fail = int((chunks_df["chunk_start"] == -1).sum())
print(f"{len(chunks_df)} chunks across {chunks_df['contract_id'].nunique()} contracts "
      f"({n_offset_fail} offset-recovery failures)")
chunks_df.head()

8031 chunks across 408 contracts (0 offset-recovery failures)


,contract_id,chunk_id,chunk_index,chunk_text,chunk_start,chunk_end
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,EXHIBIT 10.6\n\nDISTRIBUTOR AGREEMENT\n\nTHIS ...,0,3811
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,1,1.2 License. The Company hereby grants the Dis...,2862,6326
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,2,Page -2-\n\nprovided by the Company as Exhibit...,5906,9049
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,3,The aggregate units to be sold on an annual ba...,8110,12157
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,4,"Page -4-\n\nfloods, wars, sabotage, accidents ...",11634,14975


### Manual QA: eyeballed real chunk output

Automated checks (offset-recovery failures, the PRESERVED/SPLIT/OVERSIZED
audit above) verify the pipeline *ran correctly*, not that its output is
actually usable. To check that, a few real chunks from `chunks_df` /
`chunk_evidence_df` were read by hand:

- **First/last chunk of a contract** (`ASIANDRAGONGROUPINC...Reseller
  Agreement__chunk0` / `__chunk17`): both are coherent — chunk 0 starts
  cleanly at "Exhibit 10.5 / Reseller Agreement" (no truncated lead-in),
  chunk 17 ends on a complete sentence, not a mid-word cut.
- **A chunk carrying evidence** (same contract, `__chunk0`): `chunk_evidence_df`
  correctly links evidence ids `...__Document Name_0` and `...__Parties_0/_1`
  to that chunk, and each `annotation_text` sits inside a full, readable
  sentence rather than an isolated fragment — IDs and evidence both survive
  chunking intact.
- **A tricky section** (`ArmstrongFlooringInc...Intellectual Property
  Agreement`, a flattened patent/trademark schedule table): the source
  extraction already collapsed this table into one long space-separated run
  with no `\n`/`. ` separators, so the chunker falls back to plain
  word-boundary splits here. At the chunk 22 → 23 boundary the normal
  192-token overlap nearly disappears (chunk 22 ends at char 50672, chunk 23
  starts at 50674 — a 2-character *gap*, not overlap), so a table row split
  across that boundary loses its row-mates. IDs still resolve correctly (not
  a bug), but a reader looking at either chunk alone can't tell which patent
  number belongs to which filing.

**Conclusion:** IDs and evidence spans are preserved correctly in every
sample checked; prose chunks read cleanly at their boundaries. The one real
limitation is dense, delimiter-free tabular sections, where overlap can
shrink to near zero and a table row can be split without its context — a
known tradeoff of token-based chunking on this source data, not a pipeline
bug, and worth flagging rather than silently treating chunking as "done."

`chunks_df` only has chunk text and offsets — it doesn't carry `clean_df`'s
`id` / `category` / `annotation_text` / `annotation_start` at all. The task
asks to "preserve IDs and evidence," so that link has to be made explicit:
for every labeled evidence span, find every chunk whose `[chunk_start,
chunk_end)` overlaps the span's `[annotation_start, annotation_end)`, and
record whether the span sits fully inside that chunk (`fully_contained`) or
is only partially covered (the clause straddles a chunk boundary). One row
per (evidence id, overlapping chunk) pair, so a span that's split across two
chunks gets two rows.

We also classify every evidence span into `classify_spans`' PRESERVED /
SPLIT / OVERSIZED / NO_COVERAGE buckets (imported from
`chunking_util.py` rather than reimplemented, so this report
can't silently drift from the audited numbers quoted above). A naive
"matched to any chunk vs. not" count would lump spans that are too long for
any 768-token chunk (OVERSIZED) together with spans whose covering chunk
failed offset recovery upstream (NO_COVERAGE) — those are different failure
modes with different fixes, so we keep them separate here too.

In [10]:
evidenced = clean_df[~clean_df["is_impossible"]].copy()
evidenced["annotation_start"] = evidenced["annotation_start"].astype(int)
evidenced["annotation_end"] = evidenced["annotation_start"] + evidenced["annotation_text"].str.len()

matchable_chunks = chunks_df[chunks_df["chunk_start"] >= 0]

# per-(evidence id, overlapping chunk) rows -- this is the actual "link evidence to chunks" output
rows = []
for contract_id, group in evidenced.groupby("contract_id"):
    contract_chunks = matchable_chunks[matchable_chunks["contract_id"] == contract_id]
    for _, ev in group.iterrows():
        overlapping = contract_chunks[
            (ev["annotation_start"] < contract_chunks["chunk_end"])
            & (ev["annotation_end"] > contract_chunks["chunk_start"])
        ]
        for _, ch in overlapping.iterrows():
            fully_contained = (
                ev["annotation_start"] >= ch["chunk_start"] and ev["annotation_end"] <= ch["chunk_end"]
            )
            rows.append({
                "id": ev["id"],
                "chunk_id": ch["chunk_id"],
                "contract_id": contract_id,
                "category": ev["category"],
                "annotation_text": ev["annotation_text"],
                "annotation_start": ev["annotation_start"],
                "annotation_end": ev["annotation_end"],
                "fully_contained": fully_contained,
            })
chunk_evidence_df = pd.DataFrame(rows)

# per-evidence-span PRESERVED/SPLIT/OVERSIZED/NO_COVERAGE bucket, reusing the audited classifier
# instead of inferring buckets from chunk_evidence_df (which can't tell OVERSIZED from NO_COVERAGE)
spans_df = classify_spans(clean_df, chunks_df, token_len, WINDOW_SIZE)

n_evidence_total = len(spans_df)
bucket_counts = spans_df["bucket"].value_counts().reindex(
    ["PRESERVED", "SPLIT", "OVERSIZED", "NO_COVERAGE"], fill_value=0
)
print(f"{n_evidence_total} evidence spans:")
for bucket, n in bucket_counts.items():
    print(f"  {bucket:12s}: {n:5d} ({n / n_evidence_total * 100:5.1f}%)")
print(f"\n{chunk_evidence_df['id'].nunique()} spans matched to at least one chunk, "
      f"{len(chunk_evidence_df)} (id, chunk) link rows")
chunk_evidence_df.head()

11180 evidence spans:
  PRESERVED   : 11144 ( 99.7%)
  SPLIT       :    36 (  0.3%)
  OVERSIZED   :     0 (  0.0%)
  NO_COVERAGE :     0 (  0.0%)

11180 spans matched to at least one chunk, 12686 (id, chunk) link rows


,id,chunk_id,contract_id,category,annotation_text,annotation_start,annotation_end,fully_contained
0,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,Document Name,CO-BRANDING AND ADVERTISING AGREEMENT,44,81,True
1,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,Parties,2TheMart,427,435,True
2,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,Parties,"I-ESCROW, INC.",166,180,True
3,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,Parties,"2THEMART.COM, INC.",303,321,True
4,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,Parties,i-Escrow,287,295,True


## Build Chunk-Level Labels and Export

Everything above stops at "which chunks contain evidence" — that's not yet a
usable training/eval artifact for a downstream retriever or QA model, and it
never touches the categories that are `is_impossible` for a given contract
(there all chunks are true negatives, not "no data"). This section builds
one row per `(chunk, category)` for every contract, labeled positive iff
that chunk overlaps evidence for that category (from `chunk_evidence_df`)
and negative otherwise -- including `is_impossible` categories, where every
chunk in the contract is a negative by construction. It then persists
`clean_df`, `chunks_df`, `chunk_evidence_df`, and `chunk_labels_df` so this
notebook actually produces something downstream code can load instead of
ending on printed stats.

In [11]:
# one row per (contract, category), independent of is_impossible
contract_categories = clean_df[["contract_id", "category", "is_impossible"]].drop_duplicates()

# cross with every chunk in that contract
chunk_labels_df = contract_categories.merge(
    chunks_df[["contract_id", "chunk_id"]], on="contract_id"
)

# positive iff this (chunk, category) pair actually overlaps evidence; NaN (no match) -> 0
positive_keys = chunk_evidence_df[["chunk_id", "category"]].drop_duplicates()
positive_keys["label"] = 1
chunk_labels_df = chunk_labels_df.merge(positive_keys, on=["chunk_id", "category"], how="left")
chunk_labels_df["label"] = chunk_labels_df["label"].fillna(0).astype(int)
chunk_labels_df = chunk_labels_df[["contract_id", "chunk_id", "category", "is_impossible", "label"]]

# is_impossible categories should never end up with a positive label -- there is no evidence
# span to have matched against, so a 1 here would mean a real bug in the merge above
assert (chunk_labels_df.loc[chunk_labels_df["is_impossible"], "label"] == 0).all(), \
    "is_impossible category got a positive chunk label -- merge bug"

n_pos = int(chunk_labels_df["label"].sum())
print(f"{len(chunk_labels_df)} (chunk, category) pairs, {n_pos} positive ({n_pos / len(chunk_labels_df) * 100:.2f}%)")
chunk_labels_df.head()

329271 (chunk, category) pairs, 8233 positive (2.50%)


,contract_id,chunk_id,category,is_impossible,label
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,False,1
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,False,0
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,False,0
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,False,0
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,False,0


In [12]:
# clean_df carries each contract's full cleaned text once per (contract, category) row
# (~55x duplication on average) -- fine in memory, but exporting it as-is bloats the
# parquet file with the same contract text repeated dozens of times. Split it into a
# normalized contracts_df (one row per contract) plus a slim clean_df without the text
# column; join on contract_id to get the text back.
contracts_df = clean_df.drop_duplicates("contract_id")[["contract_id", "text"]].reset_index(drop=True)
clean_df_export = clean_df.drop(columns="text")

OUTPUT_DIR = Path(os.getcwd()).parent / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

contracts_df.to_parquet(OUTPUT_DIR / "contracts_df.parquet", index=False)
clean_df_export.to_parquet(OUTPUT_DIR / "clean_df.parquet", index=False)
chunks_df.to_parquet(OUTPUT_DIR / "chunks_df.parquet", index=False)
chunk_evidence_df.to_parquet(OUTPUT_DIR / "chunk_evidence_df.parquet", index=False)
chunk_labels_df.to_parquet(OUTPUT_DIR / "chunk_labels_df.parquet", index=False)

print(f"wrote contracts_df, clean_df, chunks_df, chunk_evidence_df, chunk_labels_df to {OUTPUT_DIR}")

wrote contracts_df, clean_df, chunks_df, chunk_evidence_df, chunk_labels_df to /Users/alec/Downloads/accenture-proj/Accenture-1N-contract-review-challenge/data/processed


## Baseline: TF-IDF + Logistic Regression

Quick baseline for the chunk-labeling task: TF-IDF over `chunk_text`, one
Logistic Regression classifier per category (`OneVsRestClassifier`), trained
to predict whether a chunk is positive for that category (from
`chunk_labels_df`). Split by `contract_id` (not by chunk) so chunks from the
same contract never leak across train/test.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, f1_score

# pivot chunk_labels_df (one row per chunk,category) into one row per chunk,
# with a binary column per category -- what the multi-label classifier needs
label_matrix = chunk_labels_df.pivot_table(
    index="chunk_id", columns="category", values="label", fill_value=0
)
categories = label_matrix.columns.tolist()

baseline_df = chunks_df.set_index("chunk_id")[["contract_id", "chunk_text"]].join(label_matrix)
baseline_df = baseline_df.dropna(subset=["chunk_text"])

# split by contract_id so chunks from the same contract stay on one side
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, test_idx = next(splitter.split(baseline_df, groups=baseline_df["contract_id"]))
train_df, test_df = baseline_df.iloc[train_idx], baseline_df.iloc[test_idx]
print(f"train: {len(train_df)} chunks ({train_df['contract_id'].nunique()} contracts), "
      f"test: {len(test_df)} chunks ({test_df['contract_id'].nunique()} contracts)")

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english")
X_train = vectorizer.fit_transform(train_df["chunk_text"])
X_test = vectorizer.transform(test_df["chunk_text"])
y_train = train_df[categories].values
y_test = test_df[categories].values

clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced"))
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"\nmicro F1: {f1_score(y_test, y_pred, average='micro', zero_division=0):.3f}")
print(f"macro F1: {f1_score(y_test, y_pred, average='macro', zero_division=0):.3f}")
print("\nper-category report:\n")
print(classification_report(y_test, y_pred, target_names=categories, zero_division=0))